In [ ]:
!pip install -q xgboost

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "placement_predict_50k_adjusted.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50000, 21)


,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,CGPA,AttendancePercent,Internships,...,Workshops,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,PlacementStatus,IsAnomaly
0,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.63,68.3,2,...,0.0,1,0,62.3,6.57,40.6,65.7,No,0,0
1,Male,Chennai,Tier2,ECE,AI,Yes,No,6.40,71.0,1,...,0.0,2,0,44.0,5.86,40.3,51.8,No,0,0
2,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.73,75.1,1,...,2.0,2,1,73.8,7.50,73.6,67.9,No,1,0
3,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.73,99.2,4,...,5.0,6,2,100.0,9.41,98.7,NaN,No,1,0
4,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,9.01,99.6,2,...,NaN,4,2,90.8,9.24,83.1,100.0,No,1,0


In [ ]:
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("Preprocessing completed!")
print("Features:", X.shape)
print("Target:", y.shape)

Preprocessing completed!
Features: (50000, 19)
Target: (50000,)


In [ ]:
VAL_SIZE = 0.15
TEST_SIZE = 0.15

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

Train: (34999, 19)
Validation: (7501, 19)
Test: (7500, 19)


In [ ]:
results = []

ada_base = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)

ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)

t0 = time.time()

ada.fit(X_train, y_train)

ada_fit_time = time.time() - t0

ada_val_pred = ada.predict(X_val)
ada_val_proba = ada.predict_proba(X_val)[:, 1]

results.append({
    "model": "AdaBoost",
    "val_accuracy": accuracy_score(y_val, ada_val_pred),
    "val_f1": f1_score(y_val, ada_val_pred),
    "val_roc_auc": roc_auc_score(y_val, ada_val_proba),
    "best_n_estimators": ada.n_estimators,
    "fit_time_sec": round(ada_fit_time, 2)
})

print("AdaBoost completed!")

AdaBoost completed!


In [ ]:
xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

t0 = time.time()

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_fit_time = time.time() - t0

xgb_val_pred = xgb.predict(X_val)
xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

results.append({
    "model": "XGBoost",
    "val_accuracy": accuracy_score(y_val, xgb_val_pred),
    "val_f1": f1_score(y_val, xgb_val_pred),
    "val_roc_auc": roc_auc_score(y_val, xgb_val_proba),
    "best_n_estimators": xgb.best_iteration + 1,
    "fit_time_sec": round(xgb_fit_time, 2)
})

print("XGBoost completed!")

XGBoost completed!


In [ ]:
results_df = pd.DataFrame(results).sort_values(
    "val_accuracy",
    ascending=False
).reset_index(drop=True)

print("Validation leaderboard (sorted by val_accuracy):")

display(results_df)

Validation leaderboard (sorted by val_accuracy):


,model,val_accuracy,val_f1,val_roc_auc,best_n_estimators,fit_time_sec
0,AdaBoost,0.796294,0.780963,0.880162,200,16.00
1,XGBoost,0.795894,0.782744,0.882638,135,2.63


In [ ]:
results_df.to_csv(
    "boosting_benchmark_results.csv",
    index=False
)

print("Saved results to boosting_benchmark_results.csv")

Saved results to boosting_benchmark_results.csv
